# CreaseIQ — LangGraph Workflow (v1.6 architecture)

The workflow lives in modules under `backend/src/`; this notebook imports them, builds the graph and runs demos. **All settings come from `backend/config/config.toml`** (nothing tunable is hard-coded in the code).

All settings live in one file, `backend/config/config.toml`: `[settings.*]` (switches, limits, thresholds, LLM endpoint, guardrails), `[lexicon.*]` (word lists, patterns, stub data, mock web), `[prompts]`, `[strings.*]` (English/Hindi messages) and `[kb.*]` (knowledge base: chunking, BM25, Chroma, RRF).

| Module (`src/`) | Role |
|---|---|
| `config.py` | loads `config/config.toml` |
| `state.py` | `CreaseState`, `RouteResult`, `Analysis` |
| `llm.py` | stub or OpenAI-compatible model; `llm_text`, `llm_structured` |
| `tools.py` | per-format knowledge-base search (hybrid BM25 + semantic via `build_kb.py`), `get_clause`, mock `web_search` |
| `route.py` | `load_or_initialize_state`, `decide_route`, `understand_and_route` |
| `rag.py` | retrieve → evidence check → reformulate → analyse → answer → validate → repair |
| `nodes.py` | gates, web branch, clarification, terminal handlers |
| `guardrails.py` | `sanitize_input`, `input_guard`, `output_guard`, output checks |
| `routers.py`, `graph.py` | the diamonds and the compiled `StateGraph` |
| `runner.py` | `ask()` (FastAPI stand-in) |

Set `llm.use_stub = false` in `config.toml` to use the local Ollama model. Build the knowledge base first: `.venv/bin/python src/build_kb.py build`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
SRC = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path.insert(0, str(SRC))      # the modules live in backend/src

from config import settings
from graph import build_graph, NODES, GUARDRAIL_ADDITIONS
from runner import ask

print("stub LLM:", settings.llm.use_stub, "| general Q&A:", settings.features.general_qa_enabled)

## 2. Build the graph
PII redaction is **not** in the graph: it runs in FastAPI before the graph is entered (`sanitize_input` stands in for it).

In [ ]:
graph = build_graph()
print(len(NODES), "nodes compiled")

In [ ]:
print(graph.get_graph().draw_mermaid())      # paste into any Mermaid viewer to see the compiled graph

## 3. Run it
`ask()` redacts contact PII, invokes the graph on a `thread_id` (the conversation id) and prints the node path it took.

In [ ]:
_ = ask(graph, "Is it a no ball if the bowler's front foot lands beyond the popping crease in a T20I?", "demo-1")

In [ ]:
# Clarification: the first turn asks for the format, the second turn (same thread) answers.
_ = ask(graph, "The batter left the crease and the ball hit the stumps. Is he out?", "demo-2")
_ = ask(graph, "It was an ODI and the ball was live.", "demo-2")

## Next steps
1. Extract `clause_title` / `refers_outside` at knowledge-base build time and re-tune `retrieval.min_score` .
2. Set `llm.use_stub = false` and check `understand_and_route` structured output on Ollama, then Sarvam (English and Hindi).
3. Swap `InMemorySaver` for the production checkpointer, and move `sanitize_input` into FastAPI.
4. Replace the mock `web_search` with the chosen provider.